In [3]:
import evaluate
import numpy as np
metric = evaluate.load("seqeval")
import json

<ipython-input-3-9a9bb19245b7>:3: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("seqeval")


In [4]:
import pickle 
 
with open('./../data/ner_token_id_name.pkl', 'rb') as f:
    label_list = pickle.load(f)

In [5]:
label_list

{0: 'O',
 1: 'B-PER',
 2: 'B-LOC',
 3: 'B-ORG',
 4: 'I-PER',
 5: 'I-LOC',
 6: 'I-ORG'}

In [6]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [7]:
def tokenize_and_align_labels(examples, label_all_tokens = True):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=512)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    print(len(examples))
    return tokenized_inputs

## Load Dataset

In [8]:
from datasets import Sequence

In [9]:
from datasets import Dataset, load_dataset

In [10]:
train_dataset = load_dataset("json", data_files={'train': "./../data/ner_dataset_sample.json"}, field ='train', cache_dir="./.cache", use_auth_token='d571e4a991a11177ca82035dff7a51cd8ba3f823')

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset json downloaded and prepared to /content/.cache/json/default-702dda2315770023/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
train_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'ner_tags', 'tokens'],
        num_rows: 17715
    })
})

In [12]:
map(lambda x: None if x in none_items else x, train_dataset)

In [13]:
test_dataset = load_dataset("json", data_files={'test': "./../data/ner_dataset_sample.json"}, field ='test', cache_dir="./.cache", use_auth_token='d571e4a991a11177ca82035dff7a51cd8ba3f823')

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset json downloaded and prepared to /content/.cache/json/default-93b848e14712120d/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
validation_dataset = load_dataset("json", data_files={'validation': "./../data/ner_dataset_sample.json"}, field ='validation', cache_dir="./.cache", use_auth_token='d571e4a991a11177ca82035dff7a51cd8ba3f823')

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Dataset json downloaded and prepared to /content/.cache/json/default-13c4859f71e79495/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

## Make tokenizer

In [15]:
def build_fast_bert_tokenizer(files, max_vocab_size):
    from tokenizers import decoders, models, normalizers, pre_tokenizers, processors, trainers, Tokenizer
    
    from transformers import BertTokenizerFast, AutoTokenizer, DistilBertTokenizerFast
    assert isinstance(files, list)
    tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))
    tokenizer.normalizer = normalizers.Sequence(
        [normalizers.NFD(), normalizers.Lowercase()]
    )
    tokenizer.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
    special_tokens = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
    trainer = trainers.WordPieceTrainer(vocab_size=max_vocab_size, special_tokens=special_tokens)
    tokenizer.train(files=files, trainer=trainer)
    cls_token_id = tokenizer.token_to_id("[CLS]")
    sep_token_id = tokenizer.token_to_id("[SEP]")
    tokenizer.post_processor = processors.TemplateProcessing(
        single=f"[CLS]:0 $A:0 [SEP]:0",
        pair=f"[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1",
        special_tokens=[
            ("[CLS]", cls_token_id),
            ("[SEP]", sep_token_id),
        ],
    )
    tokenizer.decoder = decoders.WordPiece(prefix="##")
    return BertTokenizerFast(tokenizer_object=tokenizer)

In [16]:
tokenizer = build_fast_bert_tokenizer(["./../data/ner_all_tokens_vocab.txt"], 30000)

In [17]:
tokenizer("মনির ঢাকায় থাকে").tokens()

['[CLS]', 'মনির', 'ঢাকায়', 'থাকে', '[SEP]']

In [18]:
tokenizer.decode(tokenizer("b_ner on the way to train!!")["input_ids"])

'[CLS] b [UNK] ner on the way to train [UNK] [UNK] [SEP]'

In [19]:
tokenizer("মনির ঢাকায় থাকে").tokens()

['[CLS]', 'মনির', 'ঢাকায়', 'থাকে', '[SEP]']

In [20]:
len(tokenizer.vocab)

30000

## Data Collator

In [21]:
batch_size = 16

In [22]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)

In [23]:
train_ds = train_dataset.map(tokenize_and_align_labels, batched=True)
test_ds = test_dataset.map(tokenize_and_align_labels, batched=True)
validation_ds = validation_dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/17715 [00:00<?, ? examples/s]

3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3


Map:   0%|          | 0/2215 [00:00<?, ? examples/s]

3
3
3


Map:   0%|          | 0/2214 [00:00<?, ? examples/s]

3
3
3


In [24]:
validation_dataset

DatasetDict({
    validation: Dataset({
        features: ['id', 'ner_tags', 'tokens'],
        num_rows: 2214
    })
})

## Train

In [25]:
from transformers import TrainingArguments

In [26]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, BertForTokenClassification

model = BertForTokenClassification.from_pretrained("neuralspace-reverie/indic-transformers-bn-bert", num_labels=len(label_list))

Some weights of the model checkpoint at neuralspace-reverie/indic-transformers-bn-bert were not used when initializing BertForTokenClassification: ['cls.predictions.transform.dense.weight', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at neuralspac

In [27]:
args = TrainingArguments(
    output_dir="Hello",
    eval_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=1,
    report_to=[],
    save_steps=2000,
)

In [28]:
trainer = Trainer(
    model,
    args,
    train_dataset=train_ds["train"],
    eval_dataset=validation_ds["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    
)

In [29]:
trainer.train()

The following columns in the training set don't have a corresponding argument in `BertForTokenClassification.forward` and have been ignored: tokens, ner_tags, id. If tokens, ner_tags, id are not expected by `BertForTokenClassification.forward`,  you can safely ignore this message.
/usr/local/lib/python3.8/dist-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 17715
  Num Epochs = 3
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 3324
  Number of trainable parameters = 133903879
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method i

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.412100,0.306025,0.559200,0.354975,0.434276,0.913017
2,0.009500,0.248428,0.567489,0.479181,0.519610,0.926152
3,0.029800,0.237650,0.590593,0.509527,0.547073,0.930630


The following columns in the evaluation set don't have a corresponding argument in `BertForTokenClassification.forward` and have been ignored: tokens, ner_tags, id. If tokens, ner_tags, id are not expected by `BertForTokenClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 2214
  Batch size = 16
Saving model checkpoint to Hello/checkpoint-2000
Configuration saved in Hello/checkpoint-2000/config.json
Model weights saved in Hello/checkpoint-2000/pytorch_model.bin
tokenizer config file saved in Hello/checkpoint-2000/tokenizer_config.json
Special tokens file saved in Hello/checkpoint-2000/special_tokens_map.json
The following columns in the evaluation set don't have a corresponding argument in `BertForTokenClassification.forward` and have been ignored: tokens, ner_tags, id. If tokens, ner_tags, id are not expected by `BertForTokenClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examp

TrainOutput(global_step=3324, training_loss=0.3022886062535165, metrics={'train_runtime': 464.2693, 'train_samples_per_second': 114.47, 'train_steps_per_second': 7.16, 'total_flos': 765009905202180.0, 'train_loss': 0.3022886062535165, 'epoch': 3.0})

## Tests

In [30]:
trainer.evaluate()

The following columns in the evaluation set don't have a corresponding argument in `BertForTokenClassification.forward` and have been ignored: tokens, ner_tags, id. If tokens, ner_tags, id are not expected by `BertForTokenClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 2214
  Batch size = 16


{'eval_loss': 0.23764966428279877,
 'eval_precision': 0.5905930470347648,
 'eval_recall': 0.5095271700776288,
 'eval_f1': 0.54707330933889,
 'eval_accuracy': 0.9306302786210213,
 'eval_runtime': 4.8096,
 'eval_samples_per_second': 460.333,
 'eval_steps_per_second': 28.901,
 'epoch': 3.0}

In [31]:
model.save_pretrained('./../model/ner_model')
tokenizer.save_pretrained('./../model/ner_tokenizer')

Configuration saved in /content/drive/MyDrive/b_ner_v2_3/ner_model/config.json
Model weights saved in /content/drive/MyDrive/b_ner_v2_3/ner_model/pytorch_model.bin
tokenizer config file saved in /content/drive/MyDrive/b_ner_v2_3/ner_tokenizer/tokenizer_config.json
Special tokens file saved in /content/drive/MyDrive/b_ner_v2_3/ner_tokenizer/special_tokens_map.json


('/content/drive/MyDrive/b_ner_v2_3/ner_tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/b_ner_v2_3/ner_tokenizer/special_tokens_map.json',
 '/content/drive/MyDrive/b_ner_v2_3/ner_tokenizer/vocab.txt',
 '/content/drive/MyDrive/b_ner_v2_3/ner_tokenizer/added_tokens.json',
 '/content/drive/MyDrive/b_ner_v2_3/ner_tokenizer/tokenizer.json')